# Rastreabilidade de Identidades na Camada Silver

Este notebook consulta o registro restrito de clientes e relaciona os dados originais, incluindo `nome`, `cpf`, `email` e `telefone`, ao identificador pseudonimizado usado na camada Silver.

O arquivo `data/restricted/customer_identity_map.parquet` contém dados pessoais de acesso restrito. Execute este notebook somente em ambiente autorizado e não compartilhe suas saídas.

In [25]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "utils").is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

raw_path = project_root / "data" / "raw" / "customers.parquet"
restricted_path = project_root / "data" / "restricted" / "customer_identity_map.parquet"
silver_path = project_root / "data" / "silver" / "customers.parquet"

print(f"Projeto: {project_root}")
print(f"Raw: {raw_path}")
print(f"Mapa restrito: {restricted_path}")
print(f"Silver: {silver_path}")

Projeto: C:\Users\Mileno\Downloads\Projeto LGPD
Raw: C:\Users\Mileno\Downloads\Projeto LGPD\data\raw\customers.parquet
Mapa restrito: C:\Users\Mileno\Downloads\Projeto LGPD\data\restricted\customer_identity_map.parquet
Silver: C:\Users\Mileno\Downloads\Projeto LGPD\data\silver\customers.parquet


In [26]:
raw = pd.read_parquet(raw_path)
identity_map = pd.read_parquet(restricted_path)
silver = pd.read_parquet(silver_path)

missing_raw_columns = [
    column
    for column in raw.columns
    if column not in identity_map.columns
]

extra_restricted_columns = [
    column
    for column in identity_map.columns
    if column not in raw.columns and column != "customer_id_protected"
]

assert not missing_raw_columns, missing_raw_columns
assert not extra_restricted_columns, extra_restricted_columns
assert len(raw) == len(identity_map)
assert "customer_id" in silver.columns
assert identity_map["customer_id"].is_unique
assert identity_map["customer_id_protected"].is_unique

for column in raw.columns:
    assert identity_map[column].equals(raw[column]), column

print(f"Raw: {len(raw):,} registros / {len(raw.columns)} colunas")
print(f"Mapa restrito: {len(identity_map):,} registros / {len(identity_map.columns)} colunas")
print(f"Silver: {len(silver):,} registros / {len(silver.columns)} colunas")
print("Todas as colunas e valores da Raw foram preservados no mapa restrito.")

Raw: 10,000 registros / 24 colunas
Mapa restrito: 10,000 registros / 25 colunas
Silver: 10,000 registros / 12 colunas
Todas as colunas e valores da Raw foram preservados no mapa restrito.


## Join de rastreabilidade

O identificador pseudonimizado é a chave de ligação: `customer_id_protected` no mapa corresponde a `customer_id` no Silver. A validação `one_to_one` impede duplicidades silenciosas.

In [27]:
silver_for_join = silver.rename(
    columns={"customer_id": "customer_id_protected"}
).copy()

identity_with_silver = identity_map.merge(
    silver_for_join,
    on="customer_id_protected",
    how="inner",
    validate="one_to_one",
    indicator=True,
)

assert len(identity_with_silver) == len(identity_map)
assert identity_with_silver["_merge"].eq("both").all()

identity_with_silver = identity_with_silver.drop(columns="_merge")

print(f"Registros relacionados: {len(identity_with_silver):,}")
print("Join concluído com correspondência total entre o mapa e o Silver.")

Registros relacionados: 10,000
Join concluído com correspondência total entre o mapa e o Silver.


In [28]:
display_columns = [
    "customer_id",
    "customer_id_protected",
    "nome",
    "cpf",
    "email",
    "telefone",
    "raca_etnia",
    "condicao_saude",
    "tipo_sanguineo",
    "estado",
    "faixa_etaria",
    "faixa_renda",
    "quantidade_compras",
    "ticket_medio",
]

available_columns = [
    column
    for column in display_columns
    if column in identity_with_silver.columns
]

display(identity_with_silver[available_columns].head(5))

,customer_id,customer_id_protected,nome,cpf,email,telefone,raca_etnia,condicao_saude,tipo_sanguineo,faixa_etaria,faixa_renda,ticket_medio
0,23b8c1e9-3924-46de-beb1-3b9046685257,9a56faa398e739ee3e38b7a6796847f96bee3182920ef1...,Srta. Ísis Borges,960.781.425-85,zoeleao@example.org,(041) 9402-6542,Preta,Infecção por HIV,O+,35-44,10.001-20.000,9275.842500
1,60e7a113-ec1b-4ca1-b91e-1d4c1ff49b78,2cdb0f55d0ae64f28f35df585624d50dc15af5ebdc4dd7...,Yuri da Rocha,953.806.174-84,guilhermegarcia@example.com,(051) 5255-3419,Indígena,Asma,O-,25-34,10.001-20.000,917.502500
2,e9c349e0-3602-48ac-90f1-bc81448aaa9e,23cc68e3025de04fcb2b10f0ea28337a676a8c820fca0e...,Luara Cunha,769.128.453-55,zpacheco@example.com,+55 21 2871-0122,Amarela,Hipertensão,O+,65+,5.001-10.000,411.330408
3,5715bd6f-a416-4293-84c2-e2e3444ea7c8,e3ad201c591c755e4b9c2858881314cc247e6b1a979c5b...,Juan Lopes,270.569.148-02,natalia93@example.org,31 2880 9570,Outro,TEA,AB+,35-44,Acima de 20.000,671.687333
4,f264accc-79ac-4b1e-a8e5-6e0c20de435d,958728b2ba9d18320d4b29becef45a730b8aa70da17a03...,Eloah da Luz,863.451.297-55,cmoreira@example.org,+55 51 3150 9839,Preta,Obesidade,AB+,18-24,Acima de 20.000,39.239560
